In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
from default_risk.scripts.auxiliar_eda_function import check_invariant

import default_risk.config as cfg
import dtale
import logging


installments_payment_df = pd.read_csv(cfg.INSTALLMENTS_PAYMENTS)

column_order_reference="DAYS_INSTALMENT"


data_frame_size=len(installments_payment_df)


installments_payment_df.sort_values(["SK_ID_PREV",column_order_reference,"DAYS_ENTRY_PAYMENT"],inplace=True)


log = logging.getLogger('werkzeug')


with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

#aux function to avoid repeated code but keep idempotence between celds
def analyze_repeated_instalments(extra_mask,case_description) :
    next_installment_number = installments_payment_df.groupby("SK_ID_PREV")["NUM_INSTALMENT_NUMBER"].shift(-1)
    rows_to_analyze = installments_payment_df[
    (installments_payment_df ["NUM_INSTALMENT_NUMBER"] == next_installment_number) 
    &
    (extra_mask)]
    print("we have " + str(len(rows_to_analyze)) + " " + case_description + "\n")
    display(rows_to_analyze.head(50))
    return rows_to_analyze

def get_full_sorted_serie_rows(rows : pd.DataFrame):
   return recreate_and_sort_series_given_rows(rows,installments_payment_df, "SK_ID_PREV",column_order_reference)

def get_full_sorted_serie_ids(ids : list):
    recreate_and_sort_the_serie_given_ids(ids,installments_payment_df, "SK_ID_PREV" ,column_order_reference)
    




Invariants:

1- the column DAYS_INSTALMENT is left capped in -2922 (100%) #4

2- the values in column DAYS_ENTRY_PAYMENT < -2952 (99.99%) #4

Soft constraints: 

1- The series starts in NUM_INSTALMENT_NUMBER = 1 or with a date older than -2890 in DAYS_INSTALMENT. Meaning that or we have register from the first installment or the first part
of the loan are beyond the temporal umbral to register data in this table. We have in train 997752 ids of previous contracts (temporal series) and just 253 series don't accomplish this rule.
(99.98%) #5
anomalies: incomplete series. 


Decisions summary: 

1- We will divide the missing values  at AMT_PAYMENT -  DAYS_ENTRY_PAYMENT in 2 diferent cases at the moment of taking metrics.

    a- Dead tails: Where the values miss and stay as nan in the remaining rows of the serie and show now relevant change in other variable. This suggest administral padding 
    so we will ignore dead tails to the compute of agg metrics.

    b- Remaining cases: Are less than 50, even if are interesting we don't have enough observations to modelate it, so we will input it with foward

2- Catch incomplete series: There is series that's does not start in NUM_INSTALMENT_NUMBER = 1 and they oldest date is < -2890 

3- For every value in DAYS_ENTRY_PAYMENT that don't fullfill the invariant #2 we will cap the value in DAYS_INSTALMENT + 30. In that way, if in the future we want to capture metrics of days of payments, that values will not destroy the proportion. 

In [4]:
#1
#files for the data dictionary
create_files_nulls_per_colmun(installments_payment_df,"installments_payment")

In [ ]:
#2
#runing screening script
eda_per_table_printing_results(installments_payment_df,schema,"installments_payments",False)

--------------------------------------
SK_ID_PREV
basic_data


,cardinality,mode
0,997752,[2360056]


frequency


,CATEGORY,COUNT,SEGMENT
0,2360056,293,top
1,2592574,279,top
2,1017477,248,top
3,1449382,243,top
4,1746731,236,top
5,1690678,223,top
6,2709164,222,top
7,1383111,220,top
8,1152155,219,top
9,2543266,216,top


--------------------------------------
SK_ID_CURR
basic_data


...

,CATEGORY,COUNT,SEGMENT
0,9000.000,254062,top
1,2250.000,179120,top
2,4500.000,174143,top
3,6750.000,173659,top
4,3375.000,149941,top
5,5625.000,96362,top
6,7875.000,60248,top
7,1125.000,60224,top
8,13500.000,42926,top
9,8100.000,37295,top


--------------------------------------
AMT_PAYMENT
basic_data


,cardinality,mode
0,944236,[9000.0]


frequency


,CATEGORY,COUNT,SEGMENT
0,9000.000,248757,top
1,2250.000,182654,top
2,4500.000,178309,top
3,6750.000,170360,top
4,3375.000,141832,top
5,5625.000,91165,top
6,1125.000,64440,top
7,7875.000,55823,top
8,13500.000,46276,top
9,8100.000,35271,top


In [6]:
#3
prev_id= installments_payment_df["SK_ID_PREV"].unique()
series_to_show= prev_id[:100]
dtale.show(get_full_sorted_serie_ids(series_to_show))

2026-05-10 19:11:20,588 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [7]:
#4
check_invariant(installments_payment_df["DAYS_INSTALMENT"] < -2922,"cases where have a older date than -2922 days", data_frame_size)

check_invariant(installments_payment_df["DAYS_ENTRY_PAYMENT"] < -2952,"cases where have a older date than -2922 days", data_frame_size)

0 of cases where cases where have a older date than -2922 days
that represent a 0.0% of cases with violation of this invariant 

820 of cases where cases where have a older date than -2922 days
that represent a 0.006027018240770706% of cases with violation of this invariant 



In [8]:
dtale.show(get_full_sorted_serie_rows(installments_payment_df[installments_payment_df["DAYS_ENTRY_PAYMENT"] < -3100]))

In [9]:
#5

grouped = installments_payment_df.groupby("SK_ID_PREV")

print(installments_payment_df["SK_ID_PREV"].nunique())

min_installment = grouped["NUM_INSTALMENT_NUMBER"].min()
min_days = grouped["DAYS_INSTALMENT"].min()

anomaly_condition = ( (min_installment > 1) & (min_days > -2890) )

anomaly_ids = anomaly_condition.index[anomaly_condition]

anomaly_rows = installments_payment_df[installments_payment_df["SK_ID_PREV"].isin(anomaly_ids)]

print(anomaly_rows["SK_ID_PREV"].nunique())

dtale.show(get_full_sorted_serie_rows(anomaly_rows))

997752
253


In [10]:
#in order to undestand the nature behind the missing values of DAYS_ENTRY_PAYMENT
rows_with_nulls=installments_payment_df[installments_payment_df["DAYS_ENTRY_PAYMENT"].isna()]
dtale.show(get_full_sorted_serie_rows(rows_with_nulls))

#in all the visualizated cases the tendence of the missing values is to show ups at the end of the secuence, like padding, so a "dead tail" definition is needed.
#Also, in the contract of SK_ID_PREV= 1004174 we can observe a jump from 8 to 101 in "NUM_INSTALMENT_NUMBER".
#This suggest that could exist errors in the counter or a sentinel / special value (101 show ups in more rows).

In [11]:
#now, let's validate the hipotesis of missing values in DAYS_ENTRY_PAYMENT y AMT_PAYMENT are dead tails. For that we will analize if exists cases
#that once the serie have a missing value in these columns, can exist a row with a value different a nan, of if once hit nan, it's always nan without relevant changes.
installments_payment_df["NEXT_VALUE_PAYMENT"] = installments_payment_df.groupby("SK_ID_PREV")["DAYS_ENTRY_PAYMENT"].shift(-1)
not_a_deadtail_mask= (installments_payment_df["DAYS_ENTRY_PAYMENT"].isna() ) & ( installments_payment_df["NEXT_VALUE_PAYMENT"].notna())
print(not_a_deadtail_mask.sum())
potencial_incosistencies_rows= installments_payment_df[not_a_deadtail_mask]
dtale.show(get_full_sorted_serie_rows(potencial_incosistencies_rows))
#When we exclude dead tail cases the remaining observation of missing values in DAYS_ENTRY_PAYMENT are less than 50. 
#Also these series exhib an anormal behaivor. Seems like have another schedule with their own counter, using the prefix "100" (101,102,103...) in NUM_INSTALLMENT_NUMBER and a different NUM_INSTALLMEMT_VERSION.



44


In [12]:
#now let's analize the behaivor of the rows when the installment number jump to 100.
groups_sizes= installments_payment_df.groupby("SK_ID_PREV")["SK_ID_PREV"].transform("size")
series_bellow_hundred_rows= installments_payment_df[groups_sizes < 100]
rows_with_notation= series_bellow_hundred_rows[series_bellow_hundred_rows["NUM_INSTALMENT_NUMBER"] > 100]
dtale.show(get_full_sorted_serie_rows(rows_with_notation)) 
#This installments with diferent numeration (100 prefix) and also have a diferent (NUM_INSTALMENT_VERSION) appears to have a high correlation with repeated "NUM_INSTALMENT_NUMBER"
#and how this usually this represent a underpayment in that month we come with the hipotesis of, this specials insatllments could be extra fees for that.


In [13]:

grouped_per_contracts= installments_payment_df.groupby("SK_ID_PREV")
contracts_with_no_repeated_installments= grouped_per_contracts["NUM_INSTALMENT_NUMBER"].nunique() == grouped_per_contracts["NUM_INSTALMENT_NUMBER"].size()
ids_contracts_to_analize= contracts_with_no_repeated_installments.index[contracts_with_no_repeated_installments]
series_with_repetead_installments= installments_payment_df[installments_payment_df["SK_ID_PREV"].isin(ids_contracts_to_analize)]
groups_sizes= series_with_repetead_installments.groupby("SK_ID_PREV").transform("size")
series_bellow_hundred_rows= series_with_repetead_installments[(groups_sizes < 80)]
rows_with_notation= series_bellow_hundred_rows[series_bellow_hundred_rows["NUM_INSTALMENT_NUMBER"] > 100]
print(len(rows_with_notation))
dtale.show(get_full_sorted_serie_rows(rows_with_notation)) 

1002


In [14]:


rows= installments_payment_df[(installments_payment_df["DAYS_INSTALMENT"] > installments_payment_df["DAYS_ENTRY_PAYMENT"])]
dtale.show(get_full_sorted_serie_rows(rows))

In [15]:
grouped_per_contracts= installments_payment_df.groupby("SK_ID_PREV")
contracts_with_no_repeated_installments= grouped_per_contracts["NUM_INSTALMENT_NUMBER"].nunique() == grouped_per_contracts["NUM_INSTALMENT_NUMBER"].size()
ids_contracts_to_analize= contracts_with_no_repeated_installments.index[~contracts_with_no_repeated_installments]
series_with_repetead_installments= installments_payment_df[installments_payment_df["SK_ID_PREV"].isin(ids_contracts_to_analize)]
dtale.show(series_with_repetead_installments)

In [18]:
#now, before end the EDA of this table we want to understand the nature behind of the repeated number of installments
rows_that_repeat_number= analyze_repeated_instalments(True,"of cases where the installment number are repeated")
dtale.show(get_full_sorted_serie_rows(rows_that_repeat_number))
#this show how repeated installments mainly represent underpayment / partial payment of a installment. So, let's check if could be generated for other reason.

we have 741703 of cases where the installment number are repeated



,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,NEXT_VALUE_PAYMENT
2618814,1000005,176456,1.0,9,-1448.0,-1484.0,14713.605,2.790,-1445.0
10749262,1000015,315553,1.0,5,-1243.0,-1278.0,5597.460,2.160,-1239.0
251801,1000019,176905,1.0,7,-2127.0,-2157.0,6072.165,0.090,-2125.0
13477241,1000041,449214,1.0,5,-2813.0,-2857.0,2357.910,133.875,-2811.0
11802025,1000041,449214,1.0,9,-2693.0,-2796.0,2357.910,147.735,-2685.0
12929880,1000041,449214,1.0,10,-2663.0,-2685.0,2357.910,147.825,-2657.0
916729,1000049,176852,1.0,1,-315.0,-326.0,4834.395,18000.000,-326.0
2408005,1000049,176852,4.0,2,-285.0,-297.0,14592.690,18000.000,-297.0
4250206,1000051,234782,1.0,8,-1511.0,-1516.0,3550.095,3546.000,-1491.0
4706039,1000051,234782,1.0,9,-1481.0,-1491.0,3550.095,3546.000,-1455.0


In [17]:
"""In order to show the cases where the instalment number tends to repeat, we create the cases of analysis 1,2,3,4 with their explanations above their respective code.
the conclusions are these: The instalment tends to repeat in events of underpayment / partial payment, for change in the version of the schedule, so the number can repeat even with 
full payment of the instalment if the schedule changes (NUM_INSTALMENT_VERSION). Beyond these dominant patterns we also found one more case where the instalment number 
can repeat and is highly correlated with what appears to be advance payment behavior. If you try to look for cases where "AMT_PAYMENT" == "AMT_INSTALMENT" and the schedule
don't change (NUM_INSTALMENT_VERSION) you will discover that there are no cases that meet these conditions at the same time (3rd analysis). But if you add that the payment can be 0 
("AMT_PAYMENT" == 0) it becomes possible to observe cases of full payment without reschedule. (4th analysis)"""

installments_payment_df["NEXT_INSTALLMENT_VERSION"] = installments_payment_df.groupby("SK_ID_PREV")["NUM_INSTALMENT_VERSION"].shift(-1)
same_version_versions_mask= (installments_payment_df ["NUM_INSTALMENT_VERSION"]==installments_payment_df["NEXT_INSTALLMENT_VERSION"])
full_installment_payment_mask= (installments_payment_df ["AMT_INSTALMENT"] == installments_payment_df["AMT_PAYMENT"])



#1- when the instalment number repeat but the installment was totaly paid
rows_repeated_installment_with_no_underpayment= analyze_repeated_instalments(full_installment_payment_mask, "of cases that repeat number of installment but it's not underpayment")

#2 - when the instalment number repeat and also the version
rows_same_version_and_installment_number= analyze_repeated_instalments(same_version_versions_mask,"of cases that repeat number of installment and version of the next row")

#3 - when the instalment number repeat, but it's not underpayment and the instalment version don't change (0 rows found, this does not happen in the dataset)
same_version_and_full_payment_mask = (same_version_versions_mask) & (full_installment_payment_mask)
rows_same_version_and_full_payment= analyze_repeated_instalments(same_version_and_full_payment_mask,"of cases that repeat number of installment and version of the next row")


#4 - the same that before but we allows to "AMT_INSTALMENT" being == 0. We want to include this because in previous analysis we discover that sometimes the number is repeated because 
#the installment was payment in advance and take another register repeating the instalment but with AMT_INSTALMENT being 0. (already paid)
full_payment_with_extra_row= (full_installment_payment_mask | (installments_payment_df["AMT_PAYMENT"] == 0))
same_version_and_full_payment_with_extra_row_mask = (same_version_versions_mask) & (full_payment_with_extra_row)
payed_in_advance_row= analyze_repeated_instalments(same_version_and_full_payment_with_extra_row_mask,"of cases that repeat instalment number for payment in advance")



installments_payment_df.drop(
    columns=["NEXT_INSTALLMENT_VERSION"],
    inplace=True
)


we have 372 of cases that repeat number of installment but it's not underpayment



,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,NEXT_VALUE_PAYMENT,NEXT_INSTALLMENT_VERSION
7065125,1001784,233119,2.0,1,-153.0,-153.0,9000.000,9000.000,-150.0,3.0
7614568,1003455,205313,2.0,1,-104.0,-104.0,40500.000,40500.000,-102.0,4.0
4071653,1004144,211055,2.0,1,-107.0,-107.0,45000.000,45000.000,-99.0,5.0
7098611,1004144,211055,5.0,1,-99.0,-99.0,11250.405,11250.405,-92.0,6.0
6137646,1007577,264913,2.0,1,-120.0,-120.0,45000.000,45000.000,-102.0,3.0
3248061,1026181,178998,2.0,1,-157.0,-157.0,18000.000,18000.000,-157.0,3.0
497426,1034283,154042,2.0,1,-48.0,-48.0,9000.000,9000.000,-39.0,3.0
3082798,1036226,181948,1.0,13,-962.0,-971.0,7489.215,7489.215,-971.0,2.0
10202369,1049831,301360,2.0,1,-266.0,-266.0,31050.000,31050.000,-240.0,3.0
1620357,1051240,147426,2.0,1,-41.0,-41.0,22500.000,22500.000,-29.0,3.0


we have 651557 of cases that repeat number of installment and version of the next row



,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,NEXT_VALUE_PAYMENT,NEXT_INSTALLMENT_VERSION
2618814,1000005,176456,1.0,9,-1448.0,-1484.0,14713.605,2.790,-1445.0,1.0
10749262,1000015,315553,1.0,5,-1243.0,-1278.0,5597.460,2.160,-1239.0,1.0
251801,1000019,176905,1.0,7,-2127.0,-2157.0,6072.165,0.090,-2125.0,1.0
13477241,1000041,449214,1.0,5,-2813.0,-2857.0,2357.910,133.875,-2811.0,1.0
11802025,1000041,449214,1.0,9,-2693.0,-2796.0,2357.910,147.735,-2685.0,1.0
12929880,1000041,449214,1.0,10,-2663.0,-2685.0,2357.910,147.825,-2657.0,1.0
4250206,1000051,234782,1.0,8,-1511.0,-1516.0,3550.095,3546.000,-1491.0,1.0
4706039,1000051,234782,1.0,9,-1481.0,-1491.0,3550.095,3546.000,-1455.0,1.0
6672947,1000051,234782,1.0,10,-1451.0,-1455.0,3550.095,3546.000,-1432.0,1.0
5973197,1000051,234782,1.0,11,-1421.0,-1432.0,3550.095,3546.000,-1406.0,1.0


we have 0 of cases that repeat number of installment and version of the next row



,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,NEXT_VALUE_PAYMENT,NEXT_INSTALLMENT_VERSION


we have 1427 of cases that repeat instalment number for payment in advance



,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,NEXT_VALUE_PAYMENT,NEXT_INSTALLMENT_VERSION
2841065,1000841,198725,0.0,29,-2271.0,-2298.0,9000.0,0.0,-2264.0,0.0
9485830,1000847,389124,0.0,3,-2301.0,-2330.0,9000.0,0.0,-2313.0,0.0
10518715,1000857,373592,0.0,4,-2445.0,-2471.0,6750.0,0.0,-2451.0,0.0
1358404,1001758,122569,0.0,8,-2402.0,-2432.0,1890.0,0.0,-2403.0,0.0
6373367,1004270,202138,0.0,28,-2279.0,-2307.0,6750.0,0.0,-2294.0,0.0
83015,1007055,155925,0.0,5,-2328.0,-2310.0,3375.0,0.0,-2291.0,0.0
3166130,1007055,155925,0.0,6,-2297.0,-2310.0,3375.0,0.0,-2291.0,0.0
2264269,1007055,155925,0.0,7,-2266.0,-2291.0,3375.0,0.0,-2261.0,0.0
235606,1007055,155925,0.0,8,-2236.0,-2261.0,3375.0,0.0,-2241.0,0.0
6704782,1007303,209013,0.0,20,-2215.0,-2245.0,3375.0,0.0,-2226.0,0.0
